# A neural network as a dynamical system: latent vector fields, attractors and other beautiful words

Hi friends! My name is Sabrina, I am an XAI/MI researcher, and this is our new tutorial! This time we
will work through the notions of a dynamical system, a vector field and its attractors — all inside
an autoencoder.

We will build on the paper [**Navigating the Latent Space Dynamics of Neural Models**](https://arxiv.org/abs/2505.22785)
(Marco Fumero, Luca Moschella, Emanuele Rodolà, Francesco Locatello, ICLR 2026). It caught my eye
**back in May 2025** and drew me in with its unusual framing. Writing it up now, in September 2026,
I am glad to see it accepted at ICLR 2026!

Our plan:

1. Define a vector field in the general sense and see what this object looks like inside a model.
2. Work out the details of the field (when it appears, how it converges, what theorems exist).
3. Learn what attractors and basins of attraction are, and how to find them — along with the subtleties.
4. Analyse how the field evolves over training.
5. See what this is good for, beyond pretty pictures.
6. Celebrate — because the five steps above are a walkthrough of an ICLR paper, and you got through it.

Step 6 is the most important one. If you are meeting these terms for the first time, getting to the
end is real work. So don't give up, read, re-read, and it will click!

Let's begin.

## **Part 0. Setup**

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.metrics import roc_auc_score

# CPU on purpose, so the numbers reproduce for everyone. For our toy setting a GPU buys nothing.
DEVICE = "cpu"
plt.rcParams.update({"font.size": 10, "figure.dpi": 110})


def seed_all(s=0):
    np.random.seed(s)
    torch.manual_seed(s)


seed_all(0)
print("Ready. torch", torch.__version__)

In [ ]:
#  Data and model

def load(n_train=10000, n_test=2000, fashion=False):
    '''MNIST (or FashionMNIST) scaled to [0,1], the first n_train / n_test examples.'''

    C = datasets.FashionMNIST if fashion else datasets.MNIST
    tr = C(root="./data", train=True, download=True, transform=transforms.ToTensor())
    te = C(root="./data", train=False, download=True, transform=transforms.ToTensor())
    Xtr = tr.data[:n_train].float().div(255.0).unsqueeze(1)
    Xte = te.data[:n_test].float().div(255.0).unsqueeze(1)

    return Xtr, tr.targets[:n_train].numpy(), Xte, te.targets[:n_test].numpy()


class AE(nn.Module):
    '''Convolutional autoencoder.'''

    def __init__(self, k):
        super().__init__()
        self.k = k
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.Flatten(), nn.Linear(64 * 7 * 7, k),
        )
        self.dec = nn.Sequential(
            nn.Linear(k, 64 * 7 * 7), nn.ReLU(), nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, 2, 1, output_padding=1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.dec(self.enc(x))      # F = D ∘ E : data to data

    def f(self, z):
        return self.enc(self.dec(z))      # f = E ∘ D : latent to latent


def train(model, Xtr, epochs=30, bs=128, lr=5e-4, wd=1e-4, seed=0, log=None):
    '''Adam, lr=5e-4, weight decay=1e-4 — the hyperparameters from Appendix D of the paper.'''
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    for ep in range(epochs):
        perm = torch.randperm(len(Xtr), generator=g)
        model.train()
        for i in range(0, len(Xtr), bs):
            b = Xtr[perm[i : i + bs]]
            loss = ((model(b) - b) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        if log is not None:
            log(model, ep)
    model.eval()
    return model


@torch.no_grad()
def test_mse(model, X, bs=512):
    model.eval()
    s = 0.0
    for i in range(0, len(X), bs):
        b = X[i : i + bs]
        s += ((model(b) - b) ** 2).mean(dim=(1, 2, 3)).sum().item()
    return s / len(X)


@torch.no_grad()
def iterate(model, z, n_iter=500, tol=1e-5, keep_traj=False):
    '''
    The iteration z_{t+1} = f(z_t) — equation (3) of the paper.

    The stopping criterion is taken from Appendix D: ||f(z_{t+1}) − f(z_t)||² < tol,
    or at most n_iter steps. Returns the stopping point and the number of steps
    to convergence (n_iter if it never converged).
    '''
    model.eval()
    traj = [z.clone()] if keep_traj else None
    # steps[i] — the step at which point i first met the criterion;
    # done[i] — whether it met it at all. This is bookkeeping for the report only:
    # the point keeps moving either way (see z = zn below).
    steps = torch.zeros(len(z), dtype=torch.long)
    done = torch.zeros(len(z), dtype=torch.bool)
    for t in range(n_iter):
        # One call on the WHOLE batch of points at once — there is no loop over points.
        zn = model.f(z)
        # We compare tol against the SQUARED step length (Appendix D),
        newly = (~done) & (((zn - z) ** 2).sum(1) < tol)
        steps[newly] = t + 1
        done |= newly
        # All points get updated, including the ones that already converged. No harm done:
        # they sit at a fixed point and go nowhere.
        z = zn
        if keep_traj:
            traj.append(z.clone())
        # We exit on the SLOWEST point of the batch,
        if done.all():
            break
    # Whatever never converged gets flagged separately
    steps[~done] = n_iter
    return (z, steps, traj) if keep_traj else (z, steps)

def dedup(Z, thr=0.99):
    '''
    Merging attractors by cosine, as in the paper: two stopping points count as
    the same attractor if cos(z1, z2) > thr. Returns the indices of the representatives.
    '''
    S = (F.normalize(Z, dim=1) @ F.normalize(Z, dim=1).T).numpy()
    alive, keep = np.ones(len(Z), dtype=bool), []
    for i in range(len(Z)):
        if not alive[i]:
            continue
        keep.append(i)
        alive &= S[i] <= thr
        alive[i] = False
    return np.array(keep)


def euclid_groups(Z, eps):
    '''A control for dedup: merging by an ABSOLUTE Euclidean threshold (single-link).'''
    return fcluster(linkage(pdist(Z.numpy()), "single"), eps, criterion="distance")


def attractor_set(Z, rel_tol=0.01):
    '''The same control, but with a RELATIVE threshold: rel_tol of the median pairwise distance.'''
    d = pdist(Z.numpy())
    lab = fcluster(linkage(d, "single"), rel_tol * np.median(d), criterion="distance")
    return np.array([np.where(lab == c)[0][0] for c in np.unique(lab)])


def find_attractors(model, Z0, n_iter=1000, tol=1e-12, eps=0.1):
    '''
    The reference ("generous") protocol: iterate with a large budget and a very strict tol,
    then merge the stopping points by the absolute threshold eps. In part 3 we will
    unpack why each of these three settings matters.
    '''
    # 1. Iterate all starting points to convergence. n_iter here is not "how many steps
    #    to take" but a ceiling — the maximum allowed. In our runs ~80 were enough.
    zs, st = iterate(model, Z0.clone(), n_iter=n_iter, tol=tol)
    # 2. The stopping points are smeared around the true fixed points, so we
    #    cluster them: two points belong together if a chain of neighbours closer than eps links them.
    lab = euclid_groups(zs, eps)
    # 3. The representative of a cluster is its MEAN, not one of the stopping points.
    A = torch.stack([zs[lab == c].mean(0) for c in np.unique(lab)])
    return A, zs, st

seed_all(0)
Xtr, ytr, Xte, yte = load(10000, 2000)
print("train", tuple(Xtr.shape), " test", tuple(Xte.shape), " values in", (Xtr.min().item(), Xtr.max().item()))

## **Part 1. The field inside the model**
### **1.1 The autoencoder and its latent space**

Let's start from the beginning. The object of this tutorial is the autoencoder.

Let's recall what an *autoencoder* is.

> An autoencoder $F_\Theta$ is a pair ($E_{\theta_1}, D_{\theta_2}$), where the encoder $E_{\theta_1}$ compresses (or expands) an input $x$ into a code $z = E(x)$, and the decoder $D_{\theta_2}$ reconstructs the image back from that code. The whole construction is trained so that the composition $$F = D \circ E : \mathcal{X} \to \mathcal{X}$$ can recover the object $x$, that is, give $F(x) \approx x$. Most often MSE is a valid choice for this, and a regularisation term can be added for good measure:

$$\mathcal{L}_{\mathrm{MSE}}(x) = \sum_{x \in \mathcal{X}} \|x - F_\Theta(x)\|_2^2 + \lambda R(\Theta)$$

where $R$ is explicit (a constraint on the parameters) or implicit (a constraint on training) regularisation.

In this setup we get:
- a clear view that defining the model takes two parts;
- a sharply separated component $z \in \mathcal{Z} \subseteq \mathbb{R}^k$ on which reconstruction, and the success of training, depend.

From here on we will care about $\mathcal{Z}$, and about all of it. We call it the **latent space**. The question we will investigate: how do we analyse the latent space when we have both halves of the model at hand?

**Why do we care about the latent space so much?**

By construction, latents have to be rich (and with good training they are). That follows from the intuition — from them, compressed and expanded, the decoder recovers the original object closely when training succeeds. And that has always made latents an important object of analysis.

As our test subject, let's train an autoencoder with a representation of size $k=2$ on 10,000 MNIST images. That size of hidden representation gives us the power to draw the vector field on a plane in its exact form — with no compression and no loss of information.

In [ ]:
seed_all(0)
t0 = time.time()
ae2 = train(AE(2), Xtr, epochs=30, seed=0)
print(f"training k=2: {time.time() - t0:.0f} s,  test MSE = {test_mse(ae2, Xte):.5f}")

with torch.no_grad():
    Ztr2 = ae2.enc(Xtr)          # latents of the training set
print("latent bounds:", Ztr2.min(0).values.tolist(), "…", Ztr2.max(0).values.tolist())

### **1.2 The usual reading: a map of objects**

**The standard setup for analysing $Z$:**

1. Ask a question: how is "something" encoded in the model? (Pick the something.)
2. Take some data (the set is chosen for the task — different classes, say, or a deliberately specific sample);
3. Push the data through the encoder and collect the representations for analysis;
4. Get the "trace" and analyse it (for instance, objects of one class usually cluster together; you can look at how one image morphs into another; and you can also move representations along directions built from differences of class centroids).

This framing focuses on the fact that:
- the latent space is defined and understood as a **map of objects**: a set of points plus our description, in the language of words, directions and centroids, of how it behaves;
- the object of interpretation is where **specific data** lands in the model, but not the model itself.

Let's look at both tricks at once: where the classes lie, and what the model knows how to draw at each point of the plane.

In [ ]:
# ── The usual reading of the latent space: a map of objects ──────────
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.4))

# left: where each class lies + centroids
sc = axL.scatter(Ztr2[:, 0], Ztr2[:, 1], c=ytr, cmap="tab10", s=5, alpha=0.5)
cent = np.stack([Ztr2[torch.tensor(ytr == d)].mean(0).numpy() for d in range(10)])
axL.scatter(cent[:, 0], cent[:, 1], s=190, c="white", ec="k", lw=1.2, zorder=5)
for d, (cx, cy) in enumerate(cent):
    axL.text(cx, cy, str(d), ha="center", va="center", fontsize=9, fontweight="bold", zorder=6)
axL.set_xlabel("$z_1$"); axL.set_ylabel("$z_2$")
axL.set_title("Points and centroids: which class lies where", fontweight="bold", fontsize=11)
fig.colorbar(sc, ax=axL, ticks=range(10), label="digit class")

# right: an atlas — decode a regular grid of codes
n_atlas = 14
qx = np.quantile(Ztr2[:, 0].numpy(), np.linspace(0.02, 0.98, n_atlas))
qy = np.quantile(Ztr2[:, 1].numpy(), np.linspace(0.02, 0.98, n_atlas))
grid_z = torch.tensor([[x, y] for y in qy[::-1] for x in qx], dtype=torch.float32)
with torch.no_grad():
    atlas = ae2.dec(grid_z).squeeze(1).numpy()
canvas = (atlas.reshape(n_atlas, n_atlas, 28, 28)
               .transpose(0, 2, 1, 3)
               .reshape(n_atlas * 28, n_atlas * 28))
axR.imshow(canvas, cmap="gray")
axR.set_xticks([]); axR.set_yticks([])
axR.set_title(f"Atlas: decoding a {n_atlas}×{n_atlas} grid of codes\n«what the model knows how to draw»",
              fontweight="bold", fontsize=11)

fig.suptitle("The usual reading of the latent space: a map of objects", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

- On the left. We push all 10,000 training images through the encoder and plot the resulting codes $z=E(x)$ as points coloured by digit class, and on top of that we place the centroids — the coordinate-wise means of the codes within each class.

- On the right. We look at the latent cloud. Along each coordinate we take 14 values and build a $14 \times 14$ grid of codes covering the latent cloud (the nodes are placed at quantiles from the 2nd to the 98th percentile of each axis). Then we decode every node and lay the 196 resulting images out on a single canvas in the same order — so neighbouring tiles correspond to neighbouring points of the latent space (hence the smooth morphing).

### **1.3 The second composition: $f = E \circ D$**

**A change of paradigm: what the paper proposed.**
The setup above told us where the data sits inside the model, but said nothing about what the model
gathered inside itself. The authors of [**Navigating the Latent Space Dynamics of Neural Models**](https://arxiv.org/abs/2505.22785)
propose to look at the AE on its own, as a system that has learned some structure. To do that, they
suggest studying the following construction:


$$f(z) = E \circ D (z), \qquad f : \mathcal{Z} \to \mathcal{Z}, \quad \mathcal{Z} \subseteq \mathbb{R}^k$$


Yes, the encoder of the decoder. By definition this is a map of the latent space **into itself**, and
since it maps into itself, we can repeat it as many times as we like: $f(f(f(\dots f(z))))$.

<details>
<summary><b>Visualising the round trip through $D(E(x))$</b></summary>

The notation $E \circ D$ — "the encoder of the decoder" — looks inside out. Visually it works like this:

```text
  space:   Z = R^k ── D ──▶   X = R^784    ── E ────▶   Z = R^k
  point:      z    ───────▶       D(z)     ────────▶   f(z) = E(D(z))
              └───── we left from here and came back here ──────┘
```

The decoder is $D : \mathcal{Z} \to \mathcal{X}$, the encoder is $E : \mathcal{X} \to \mathcal{Z}$. So the composition $E \circ D$ starts in $\mathcal{Z}$, goes out into $\mathcal{X}$ and returns to $\mathcal{Z}$. Its domain and codomain are therefore **the same set**, and only because of that can the result be fed to the very same function again.

For the composition $F = D \circ E$ the types also match ($\mathcal{X} \to \mathcal{X}$), and it can be iterated too. That is a variant of the classical setup:

```text
  f = E ∘ D :  Z ──▶ X ──▶ Z     analysis in the latent space
  F = D ∘ E :  X ──▶ Z ──▶ X     analysis in (approximated) data
```

</details>

Note that:

- such a representation is bounded by exactly what the network has learned inside itself, and will therefore be worse the worse our AE is;
- we can end up with the case $f(f(z)) = f(z)$ (a fixed point). In the $D(E(x)) = D(z)$ setting such points are more of an overfitting artifact and usually do not arise, whereas here they are simply points, since we do not know what the space inside the encoder looks like (and we want to find out).

From here on we denote **fixed points** by $z^* = f(z^*)$.

<details>
<summary><b>The near-identity of $f$ as a side effect of training</b></summary>

Note that the loss requires $D(E(x)) \approx x$. Take a fixed image $x'$ from the test set with representation $z' = E(x')$. For it, by definition, $D(z') \approx x'$ holds if the model is well trained. Substitute this into the composition we are studying, $f = E(D(z))$:

$$f = E\big(D(E(x'))\big) =  E(D(z')) = \approx E(x') = z'$$

so **on the latents of the training, validation and test data $f$ is almost the identity.** It is important to pin down, however, that:

- **"Almost" means exactly up to the quality of the model.** Even if the reconstruction error is around $0.05$, the residual — small at first — accumulates with every application of $f(z)$.
- **Beyond all of $E(\mathcal{X})$** there remains a (rather large) subset of $\mathcal{Z}$ that never appeared in the data (and may not exist there at all — an as yet uninvented MNIST font, say). On it nothing stops $f$ from behaving however it likes, because the loss only reached there indirectly.

>  here the thesis shows up once more — the composition $D \circ E$ is what the model **was trained on**. The composition $E \circ D$ appears nowhere in the loss: its behaviour on the latents of the data came out as a consequence, and outside the data it was not controlled at all. And everything we see in such a field is, by construction, a **side effect** of training rather than its goal.

</details>

### **1.4 Mapping what we have seen onto the mathematical machinery**

So, we have a map into itself together with a sequence of steps. In the special case, mathematics has
a matching notion — the **dynamical system**. Informally, a dynamical system is always a set of states
for which a transition between them holds, governed by a time parameter (or its analogue). Formally:

>A **dynamical system** defined on a (smooth manifold) $X$ is a map $g : \mathbb{R} \times X \to X$, written in the parametric form $g^t(x)$, where $t \in \mathbb{R}$, $x \in X$; it is differentiable, and $g^0$ is the identity map of $X$.

Smoothness is what we need for differentiability, and the manifold is what lets us see how close our $X$ is to something linear.

<details>
<summary><b> A theoretical note. </b></summary>


On top of that: for **stationary invertible** systems the family $\{g^t : t \in \mathbb{R}\}$ forms a **group of transformations**, that is $g^{t_1} \circ g^{t_2} = g^{t_1 + t_2}$. $X$ itself is then called the **phase space**, and the trajectory of a point under $g^t$ is called a **phase trajectory**.

**Stationarity** means that the rule of motion does not depend on what time it is (on which step it is): $g^t$ is determined only by the duration $t$, not by the moment of departure.
**Invertibility** means that one can also move backwards, for $t > 0$.

</details>
Inside a dynamical system two "subtypes" are allowed, depending on the form of the parameter $t$. They are:

1. **Flow** — the parameter $t$ is treated as continuous time. This definition describes an ODE (an ordinary differential equation).
2. **Cascade** — time goes in steps, and the system is given by a pair $(X, f)$, where $f : X \to X$. The trajectory of a point is the sequence $x,\ f(x),\ f^2(x), \dots$

To map the definition onto our problem, let's decompose it:

| In the definition of a flow | In our case |
|---|---|
| phase space $X$ | the latent space $\mathcal{Z} = \mathbb{R}^k$ |
| smooth manifold | $\mathbb{R}^k$ is a manifold, and a flat one: the condition holds for free |
| the map $g^t$ | $f^t = \underbrace{f \circ \dots \circ f}_{t \text{ times}}$, where $f = E \circ D$ |
| time $t \in \mathbb{R}$ | $t \in \{0, 1, 2, \dots\}$ — how many times we applied $f$ |
| $g^0 = \mathrm{id}$ | $f^0 = \mathrm{id}$: zero applications change nothing |
| $g^{t_1} \circ g^{t_2} = g^{t_1 + t_2}$ | $f^n \circ f^m = f^{n+m}$ — but only for $n, m \geq 0$ |
| the family $\{g^t\}$ forms a **group** | $\{f^n\}$ forms only a **semigroup**: there is no inverse element |
| $g$ is differentiable, including in $t$ | There is no derivative in time, but there is a derivative of $f$ in $z$ — the Jacobian $J_f$ |

Rows 1, 2, 3, 5 and 6 match. The seventh does not, because $f$ is not invertible: an inverse element for $f$ does not exist by construction, since $E$ crushes $\mathbb{R}^{784}$ into $\mathbb{R}^k$ and different points become indistinguishable after a step (there is no one-to-one correspondence).

The fourth and the eighth diverge because time is discrete, so the ratio of the increment of the function to an infinitesimal increment of the argument ($t$) is undefined (the increment is always of fixed size for us).

**Hence, when we work with the transformations described above, we are in the cascade case and we work with discrete time.**

### **1.5 Steps in space: discrete versus continuous**


- When we move **discretely**, we jump along the whole vector at once. There is no derivative in time: the sequence $z_0, z_1, z_2, \dots$ is not a function of a continuous argument, and there is nothing to take $\partial / \partial t$ of. Its analogue is the **finite difference**:

$$\Delta z_t \;=\; z_{t+1} - z_t \;=\; f(z_t) - z_t$$

- When we move **continuously**, travelling along the vector and recomputing the direction infinitely often. Here the derivative does appear, and we get an ordinary differential equation:

$$\frac{\partial z}{\partial t} \;=\; f(z) - z$$

The right-hand sides of the equations are **the same** — it is one and the same field. Only the left-hand side differs: a finite difference versus a derivative. The continuous version is the limit of the discrete one as the step length $h \to 0$, while our iteration is the opposite extreme: $h = 1$, the crudest step available. Formally this is Euler's method.

<details>
<summary><b>Aside: Euler's method</b></summary>

Euler's method approximates the solution of $\partial z / \partial t = V(z)$ like this: $z(t+h) \approx z(t) + h\,V(z(t))$. Substitute $h = 1$ and $V(z) = f(z) - z$:

$$z_{t+1} \;=\; z_t + \big(f(z_t) - z_t\big) \;=\; f(z_t)$$

— which is exactly our iteration, where $h = 1$ means there is no smallness of the step, and hence:

**The trajectories differ.** A point that has taken $t$ steps of the iteration is **not** the point the flow would have carried it to in time $t$. We are computing a polyline, not the curve of the flow.

**The fixed points, however, which we will discuss below, are the same ones.** The condition $V(z^*) = 0$ is equivalent to $f(z^*) = z^*$, with no approximation involved. So **the paths differ but the endpoints coincide** — and that is precisely why looking for what lives in the continuous picture by means of the iteration is legitimate.

</details>

Thanks to the steps, we can try to reconstruct the future of every point. The object that defines it is the vector $V(z) \;=\; f(z) - z$, and the collection of all such vectors, one at every point of the space, is called a **vector field**.

**For continuous time** the $V(z)$ introduced above is a **velocity**. It describes in which direction and how fast the point is moving right now. The future of the point is the solution of the equation $\partial z / \partial t = V(z)$: a smooth curve that is tangent to the field at each of its points. The path travelled in time $t$ is exactly the $g^t(z)$ from the definition above.

**For discrete time** it is an **increment**: where the point lands in exactly one step.

$$z_{t+1} \;=\; z_t + V(z_t) \;=\; f(z_t)$$

The future of the point is the sequence $z,\ f(z),\ f^2(z), \dots$, that is, a polyline made of finite segments. Nothing happens between the nodes: the point has no intermediate positions.

### **1.6 Fixed points and attractors**

Note that among all points there may be ones with $V(z^*) = 0$, that is $V(z^*) = f(z^*) - z^* = 0$,
$f(z^*) = z^*$. Such points are called fixed, and they can be reached **from different sides of the field**. If the trajectories of all sufficiently close points converge to $z^*$, we will call it an **attractor**.


Good. The definitions are in our hands, and it is time to look at them in our AE. Let's take 2600 starting points and look for trajectories with a naive approximation of the attractors. To do that we apply the iteration $f(z) \to z$ to each point. The resulting points (where $z$ went after the function) we merge by the Euclidean norm at 0.1 — these will be our candidate fixed points.

In [ ]:
# ── The latent vector field V(z) = f(z) − z ──────────────────────────
# What we are after: at every point z, a vector showing where one pass
# through the decoder and back through the encoder will move it. There is NO
# iteration here, we take exactly one step.

# A rectangle covering the whole latent cloud with a margin of 8 units:
# we also care about what happens outside the data.
lo = (Ztr2.min(0).values - 8).numpy()
hi = (Ztr2.max(0).values + 8).numpy()


def field_grid(model, lo, hi, n):
    """Computes the field at the nodes of an n×n grid: returns node coordinates and V(z) at each."""
    gx, gy = np.linspace(lo[0], hi[0], n), np.linspace(lo[1], hi[1], n)
    GX, GY = np.meshgrid(gx, gy, indexing="xy")
    G = torch.tensor(np.stack([GX.ravel(), GY.ravel()], 1), dtype=torch.float32)
    with torch.no_grad():
        V = (model.f(G) - G).numpy()      # the field: one step of f, minus the original point
    return GX, GY, G, V


# A coarse 26×26 grid — for the arrows (on a dense one they smear into mush).
GX, GY, G, V = field_grid(ae2, lo, hi, 26)
N = np.linalg.norm(V, axis=1).reshape(GX.shape)   # vector length at each node → we colour by it

# ── Attractors ───────────────────────────────────────────────────────
# What we are after: points that the map f does not move, z* = f(z*).
# How we look: iterate z ← f(z) to convergence and see where the trajectories
# stop. We start both from the grid and from real data codes —
# otherwise we risk missing a basin that lies outside the cloud.
GXf, GYf, Gf, _ = field_grid(ae2, lo, hi, 40)
starts = torch.cat([Gf, Ztr2[:1000]], 0)          # 1600 grid nodes + 1000 data latents

# find_attractors: iterates until the step is < 1e-12 (a ceiling of 1000 iterations is the safety net),
# then merges the stopping points by the absolute Euclidean threshold 0.1.
A2, zs_all, steps_all = find_attractors(ae2, starts)

print(f"out of {len(starts)} starting points we got {len(A2)} distinct fixed points:")
print(f"steps to convergence: median {steps_all.float().median():.0f}, max {steps_all.max()}\n")

# For each point found we check THREE different things:
#   |lambda|max < 1  — it is an attractor (definition 2: it attracts its neighbours);
#   ||J||_sigma      — whether the CONTRACTION condition holds (definition 3, which is stricter);
#   residual         — whether we really are at a fixed point rather than near one.
for i, z in enumerate(A2):
    J = torch.autograd.functional.jacobian(lambda zz: ae2.f(zz.unsqueeze(0)).squeeze(0), z.clone())
    ev = torch.linalg.eigvals(J).abs().numpy()
    with torch.no_grad():
        res = (ae2.f(z.unsqueeze(0)).squeeze(0) - z).norm().item()
    print(f"  z*_{i+1} = {z.numpy().round(2)}   |lambda|max = {ev.max():.3f}   "
          f"||J||_sigma = {torch.linalg.matrix_norm(J, 2).item():.3f}   residual = {res:.1e}")

# ── The figure ───────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.4))

# Left: colour = vector length, white lines = streamlines (where it would flow under
# continuous motion), stars = the attractors we found.
im = ax1.pcolormesh(GX, GY, N, cmap="viridis", shading="auto")
GXs, GYs, _, Vs = field_grid(ae2, lo, hi, 40)     # streamlines are drawn on the dense grid
ax1.streamplot(GXs[0], GYs[:, 0], Vs[:, 0].reshape(40, 40), Vs[:, 1].reshape(40, 40),
               color="white", linewidth=0.6, density=1.1, arrowsize=0.7)
ax1.scatter(A2[:, 0], A2[:, 1], s=180, marker="*", c="red", ec="k", lw=0.8, zorder=6, label="attractors")
ax1.set_title("Field norm $\\|V(z)\\|$ and streamlines", fontweight="bold")
ax1.legend(loc="lower right", fontsize=8)
fig.colorbar(im, ax=ax1, label="$\\|f(z)-z\\|$")

# Right: the same vectors as arrows on top of the training-set latents.
# angles/scale_units="xy" — the arrow length is proportional to the TRUE length
# of the vector rather than normalised: otherwise the main effect would vanish (the field is weak
# inside the data cloud and strong outside it). scale=1.6 just shortens them all at once.
sc = ax2.scatter(Ztr2[:, 0], Ztr2[:, 1], c=ytr, cmap="tab10", s=4, alpha=0.45)
ax2.quiver(GX, GY, V[:, 0].reshape(GX.shape), V[:, 1].reshape(GX.shape),
           angles="xy", scale_units="xy", scale=1.6, width=0.0032, color="k", alpha=0.75)
ax2.scatter(A2[:, 0], A2[:, 1], s=180, marker="*", c="red", ec="k", lw=0.8, zorder=6)
ax2.set_title("The same field on top of the training-set latents", fontweight="bold")
fig.colorbar(sc, ax=ax2, label="digit class", ticks=range(10))

for ax in (ax1, ax2):
    ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")
fig.suptitle("The latent vector field of an autoencoder, $k=2$, MNIST", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

What the picture gives us:

- **Structure.** Almost everywhere the arrows point inwards, into the region where the latents of the training images lie.

- **Places where the field vanishes (the white areas).** There $f(z) \approx z$: encoding and decoding stop changing anything. Our model has **two** such points, and both are attractors by definition 2 (the residual $\|f(z^*) - z^*\|$ is of order $10^{-5}$ and $10^{-6}$).

- **The norm of the field grows with distance from the data.** The further a point is from what the model has seen, the stronger its motion — or, put differently, the length of the arrow is a measure of "how alien this point is". In the "practical applications" block this consequence is turned into an OOD detector.

## **Part 2. Why do we converge to (almost) stillness?**

The working hypothesis: because neural networks are contractive.

> A **contractive** map is one that **brings any two points closer**: if we take $z_1$ and
> $z_2$ and apply $f$ to them, the distance between them is guaranteed to shrink, and by no less
> than a fixed factor. Convergence follows: over $t$ steps the distance falls as $C^t$, that
> is, geometrically, and any two trajectories sooner or later glue together.

We can check "how contractive a particular model is in a particular region of the data". For that we will need three definitions:

### **2.1 Definitions**

**Definition 1 (Lipschitz continuity).** A function $f : \mathcal{Z} \to \mathcal{Z}$ is Lipschitz continuous if there exists a constant $C$ such that for any pair of points
$$d(f(z_1), f(z_2))_{\mathcal{Z}} \le C \, d(z_1, z_2)_{\mathcal{Z}}$$

When $C < 1$ the map is called **contractive** (so the constant is our first indicator of contractivity).

**Definition 2 (attractor).** A fixed point $z = f(z)$ of a differentiable map $f$ is called an attractor if all eigenvalues of the Jacobian $J_f$ at that point are strictly less than one in absolute value.

**Definition 3 (the Lipschitz constant via the Jacobian).** For a differentiable $C$-Lipschitz $f$:
$$C = \sup_{z \in \mathcal{Z}} \|J_f(z)\|_\sigma$$
where $\|\cdot\|_\sigma$ is the spectral norm (the largest singular value).

The set of initial conditions $z_0$ leading to one and the same $z^*$ is called the **basin of attraction**.

A note:

- Definition 2 talks about the **eigenvalues** of the Jacobian (the spectral radius $\rho(J)$);
- Definition 3 talks about the **spectral norm** $\|J\|_\sigma$ (the largest singular value).

For non-symmetric matrices $\rho(J) \le \|J\|_\sigma$ always holds, and the gap can be arbitrarily
large. In practice this means: a point can be an **attractor** (all $|\lambda| < 1$, trajectories eventually converge to it) and at the same time **fail the contraction condition** ($\|J\|_\sigma > 1$, that is, on a single step the distance may actually grow).

### **2.2 Why neural networks are contractive, empirically**

This is because training exerts pressure towards the map behaving precisely as a
contraction. The paper lists four mechanisms:

- **Initialisation.** Standard schemes (LeCun/Glorot/He) preserve the variance of activations under idealised assumptions — i.i.d. inputs and weights. In reality the data is correlated, and residual connections break the independence of the weights, so networks at initialisation end up biased towards contraction.

- **Explicit regularisation.** Weight decay penalises the norms of the weights. For 1-Lipschitz activations
  $\|J_\Theta(x)\|_2 \le \prod_\ell \|W_\ell\|_2 \le \prod_\ell \|W_\ell\|_F$ — so the $L_2$ penalty presses directly on the product of the spectral norms (this fact is worked through in Appendix C of the paper).

- **The bottleneck.** $k = \dim(\mathcal{Z})$ constrains the rank:
  $\mathrm{rank}(J_E) \le k$, and by the chain rule this constrains the Jacobian of the whole model too.

- **Implicit regularisation.** Augmentations (noise in DAEs, masking in MAEs) penalise sensitivity to perturbations **locally**, around the training examples.

Let's check the first item: take an **untrained** model and see what it does to the variance of Gaussian noise in the latent space (this is a replication of the experiment from Appendix B.1 of the paper).

In [ ]:
# ── Contractivity at initialisation: sigma_out / sigma_in ────────────
print("ratio of output to input variance for f(z) = E(D(z)), untrained model:")
print(f"{'k':>6} | {'sigma_out / sigma_in':>22}")
for k in [2, 16, 128]:
    ratios = []
    for s in range(5):                       # 5 different initialisations
        torch.manual_seed(100 + s)
        m = AE(k).eval()
        z = torch.randn(1000, k)
        with torch.no_grad():
            out = m.f(z)
        ratios.append((out.std() / z.std()).item())
    print(f"{k:>6} | {np.mean(ratios):>13.4f} ± {np.std(ratios):.4f}")

The ratio is of order $0.03$–$0.05$, that is, **at initialisation the map $f$ contracts the space
by roughly a factor of 20–30 in a single step**. In the paper the same check is run on 12 torchvision
backbones (Figure 6): all of them have a ratio below one. So a freshly initialised network always has
an attractor, and it is the point near zero into which everything collapses.

Now let's look at what happens to contractivity **after** training.

In [ ]:
#  The local spectral norm of the Jacobian for the TRAINED model
def jac(model, z):
    '''The Jacobian J_f at the point z (for k=2 this is a 2x2 matrix, computed exactly).'''
    return torch.autograd.functional.jacobian(
        lambda zz: model.f(zz.unsqueeze(0)).squeeze(0), z.clone()
    )

def spec(J):
    return torch.linalg.matrix_norm(J, 2).item()

rs = np.random.RandomState(0)
idx = rs.choice(len(Ztr2), 200, replace=False)
on_data = np.array([spec(jac(ae2, Ztr2[i])) for i in idx])

far = torch.tensor(np.random.RandomState(1).uniform(-120, 120, size=(200, 2)), dtype=torch.float32)
off_data = np.array([spec(jac(ae2, far[i])) for i in range(200)])

for name, v in [("on the training-data latents", on_data), ("far from the data", off_data)]:
    print(f"{name:>30}: mean {v.mean():.3f}  median {np.median(v):.3f}  "
          f"max {v.max():.3f}  fraction of values < 1: {np.mean(v < 1):.2f}")

fig, ax = plt.subplots(figsize=(7, 3.4))
bins = np.linspace(0, 2.0, 45)
ax.hist(on_data, bins=bins, alpha=0.65, label="on the data", color="tomato")
ax.hist(off_data, bins=bins, alpha=0.65, label="far from the data", color="royalblue")
ax.axvline(1.0, color="k", ls="--", lw=1.2, label="contraction boundary $\\|J_f\\|_\\sigma = 1$")
ax.set_xlabel("$\\|J_f(z)\\|_\\sigma$"); ax.set_ylabel("frequency")
ax.set_title("The trained model contracts the space where the data is not", fontweight="bold", fontsize=11)
ax.legend(fontsize=8)
plt.show()

In [ ]:
# ── A map of the Jacobian spectral norm: where there is no contraction ─────────
# Let's compute
# ||J_f(z)||_sigma at every grid node and at every training-set latent, and colour the data.
from torch.func import jacrev, vmap

def spec_norm_map(model, Z, chunk=2000):
    '''||J_f||_sigma for a whole batch of points at once: vmap computes all Jacobians in one go.'''
    out = []
    jac_fn = vmap(jacrev(lambda z: model.f(z.unsqueeze(0)).squeeze(0)))
    for i in range(0, len(Z), chunk):
        J = jac_fn(Z[i : i + chunk])
        out.append(torch.linalg.matrix_norm(J, 2).detach())
    return torch.cat(out)

n_j = 50
gxj, gyj = np.linspace(lo[0], hi[0], n_j), np.linspace(lo[1], hi[1], n_j)
GXj, GYj = np.meshgrid(gxj, gyj, indexing="xy")
Gj = torch.tensor(np.stack([GXj.ravel(), GYj.ravel()], 1), dtype=torch.float32)

S_grid = spec_norm_map(ae2, Gj).numpy().reshape(n_j, n_j)   # over the grid
S_data = spec_norm_map(ae2, Ztr2).numpy()                   # at the latents themselves

# Three zones, so that "on the data" and "far from the data" do not get lumped together:
# right past the edge of the cloud the picture is already different from the far periphery.
rs_z = np.random.RandomState(0)
P = torch.tensor(rs_z.uniform(lo, hi, size=(4000, 2)), dtype=torch.float32)
in_bb = ((P[:, 0] > Ztr2[:, 0].min()) & (P[:, 0] < Ztr2[:, 0].max()) &
         (P[:, 1] > Ztr2[:, 1].min()) & (P[:, 1] < Ztr2[:, 1].max()))
far = torch.tensor(np.random.RandomState(1).uniform(-120, 120, size=(4000, 2)), dtype=torch.float32)

zones = [("training-set latents", S_data),
         ("nearby, but outside the cloud", spec_norm_map(ae2, P[~in_bb]).numpy()),
         ("far away (a ±120 box)", spec_norm_map(ae2, far).numpy())]
print(f'{"zone":<32} {"n":>6} {"median":>9} {"share < 1":>10}')
for name, v in zones:
    print(f"{name:<32} {len(v):>6} {np.median(v):>9.3f} {np.mean(v < 1):>10.2f}")

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5))

im = axA.pcolormesh(GXj, GYj, S_grid, cmap="magma", shading="auto", vmin=0, vmax=2)
axA.contour(GXj, GYj, S_grid, levels=[1.0], colors="cyan", linewidths=1.4)
axA.scatter(Ztr2[:, 0], Ztr2[:, 1], s=1.5, c="white", alpha=0.20)
axA.set_title("$\\|J_f(z)\\|_\\sigma$ across the whole space\ncyan contour — the contraction boundary $\\|J_f\\|_\\sigma = 1$",
              fontweight="bold", fontsize=11)
fig.colorbar(im, ax=axA, label="$\\|J_f\\|_\\sigma$")

sc = axB.scatter(Ztr2[:, 0], Ztr2[:, 1], c=S_data, cmap="coolwarm", vmin=0.5, vmax=1.5, s=5)
axB.scatter(A2[:, 0], A2[:, 1], s=220, marker="*", c="yellow", ec="k", lw=0.9, zorder=6)
axB.set_title("The same latents, coloured by $\\|J_f\\|_\\sigma$\nred — no contraction",
              fontweight="bold", fontsize=11)
fig.colorbar(sc, ax=axB, label="$\\|J_f\\|_\\sigma$")

for ax in (axA, axB):
    ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")
fig.suptitle("Where the trained model contracts, and where it does not", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

Note that contractivity is not binary (present or absent) but rather continuous in the distance from the data:

- **On the training-set latents themselves** there is **no** contraction at most points: the median
  $\|J_f\|_\sigma \approx 1.08$, and the condition $< 1$ holds at only 40% of points.
- **Right past the edge of the cloud** it does appear: the median is $\approx 0.69$, the condition holds at 87%.
- **On the far periphery** the model is firmly contractive: the median is $\approx 0.27$, the condition holds at 96%.

This follows from training. On the data the autoencoder is obliged **not to
lose information** — otherwise the reconstruction falls apart. A map that preserves information is locally
close to an isometry, that is, $\|J_f\| \approx 1$. The model can afford to contract to nothing only
where there is nothing of value — that is, **outside** the support of the data.

> Let's keep this result in mind: contraction lives **on the approach** to the data, not inside it.

## **Part 3. How to find an attractor**

Above we found two fixed points on the basis of motion: we iterated $f(f(f(\dots f(z)\dots)))$ until it stopped and merged the resulting points by the Euclidean norm.

Our points came out of one particular configuration: so many starts, such an iteration budget, such a merging threshold. Change any of them and the answer may change — which raises the question: how does one search optimally?

In the paper the authors give the following algorithm (Appendix D):

1. Take starting points $z_0$ (latents of training data, latents of the test set, or simply Gaussian noise).
2. Iterate $z_{t+1} = f(z_t)$ until $\|f(z_{t+1}) - f(z_t)\|_2^2 < 10^{-6}$ (or $10^{-5}$ for
   the pretrained models in sections 4.1–4.2 of the paper), but for no more than $t = 3000$ steps ($500$ respectively).
3. Count two stopping points as **the same** attractor if
   $\cos(z_1^*, z_2^*) > 0.99$.

Here each of the three steps also contains a threshold, which will also affect the answer itself. We cannot do without thresholds in the steps of the attractor search, and so limitations catch up with us:

Deduplication by cosine or by norm assumes that "two points are the same attractor if the angle/norm between them is small". But in this framing it is important to remember that we inherit the limitations of the metrics as well:

- **The cosine does not see the norm.** The points $(1, 0)$ and $(100, 0)$ have $\cos = 1$ and will be declared one attractor, even though
the distance between them is 99. For latent spaces, which are almost always anisotropic and have a preferred "mean
direction", this means that attractors differing mainly in length will collapse into one.

— **The norm is relative** and needs rescaling to the typical norm of the space.

- **The threshold is relative to the dimensionality.** In $\mathbb{R}^{128}$ two
random vectors have $\cos \approx 0$, and the threshold $0.99$ turns out to be strict: any two
stopping points differing by fractions of a percent will be counted as **different** attractors.
In $\mathbb{R}^2$ the same threshold is, conversely, generous.

- **The stopping criterion**: $\|f(z_{t+1}) - f(z_t)\|^2 < \text{tol}$ means "we are moving slowly". For linear dynamics with
contraction factor $\rho$ the residual error relates to the step size as $$\|z_t - z^*\| \approx \frac{\|z_{t+1} - z_t\|}{1 - \rho}$$

that is, at $\rho = 0.83$ (for our two attractors $\|J\|_\sigma$ equals $0.83$ and $0.66$) the error is
about **six times** larger than the last step. We stop not at the attractor but in a cloud
around it — and the size of that cloud determines how many "different" attractors we will count afterwards.

The consequence: **"the number of attractors" is not a property of the model, it is a property of
(the set of starting points, the constraints on the iteration, the criterion for merging points)**. Any claim of the form "the model now has
more attractors" is meaningful only when the conditions of the search are reported alongside it.

And besides the count, it is also useful to analyse the quality. Basins can give us that.

### **3.1 Basins of attraction**

Recall that the basin of attraction is the set of initial conditions $z_0$ leading to one and the same $z^*$.

In [ ]:
# ── Basins of attraction and decoded attractors (k=2) ────────────────
n_b = 50
gxb, gyb = np.linspace(lo[0], hi[0], n_b), np.linspace(lo[1], hi[1], n_b)
GXb, GYb = np.meshgrid(gxb, gyb, indexing="xy")
Gb = torch.tensor(np.stack([GXb.ravel(), GYb.ravel()], 1), dtype=torch.float32)

zb, sb = iterate(ae2, Gb.clone(), n_iter=120, tol=1e-8)

# which of the attractors found in part 1 the point ends up at
basin = torch.cdist(zb, A2).argmin(1).numpy().reshape(n_b, n_b)
frac = np.bincount(basin.ravel(), minlength=len(A2)) / basin.size
print("share of area per basin:", frac.round(3))

fig = plt.figure(figsize=(13.5, 5))
gs = fig.add_gridspec(4, 4, width_ratios=[2.4, 2.4, 1, 1], hspace=0.45, wspace=0.3)

ax = fig.add_subplot(gs[:, 0])
ax.pcolormesh(GXb, GYb, basin, cmap="Pastel1", shading="auto")
ax.scatter(Ztr2[:, 0], Ztr2[:, 1], s=2, c="k", alpha=0.18)
ax.scatter(A2[:, 0], A2[:, 1], s=250, marker="*", c="red", ec="k", lw=1, zorder=6)
ax.set_title("Basins of attraction", fontweight="bold", fontsize=11)
ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")

ax = fig.add_subplot(gs[:, 1])
im = ax.pcolormesh(GXb, GYb, sb.numpy().reshape(n_b, n_b), cmap="cividis", shading="auto")
ax.scatter(A2[:, 0], A2[:, 1], s=250, marker="*", c="red", ec="k", lw=1, zorder=6)
ax.set_title("Number of steps to convergence", fontweight="bold", fontsize=11)
ax.set_xlabel("$z_1$")
fig.colorbar(im, ax=ax)

with torch.no_grad():
    X_star = ae2.dec(A2)
n_show = min(len(A2), 4)
for i in range(n_show):
    a = fig.add_subplot(gs[i, 2])
    a.imshow(X_star[i, 0], cmap="gray"); a.axis("off")
    a.set_title(f"$D(z^*_{i+1})$", fontsize=9)

# the closest training image to each decoded attractor
Ftr_n = F.normalize(Xtr.flatten(1), dim=1)
sims = F.normalize(X_star.flatten(1), dim=1) @ Ftr_n.T
for i in range(n_show):
    a = fig.add_subplot(gs[i, 3])
    j = sims[i].argmax().item()
    a.imshow(Xtr[j, 0], cmap="gray"); a.axis("off")
    a.set_title(f"closest, cos={sims[i, j]:.2f}", fontsize=8)

fig.suptitle("Attractors of the $k=2$ autoencoder: basins, speed and prototypes",
             fontsize=13, fontweight="bold")
plt.show()

What matters here:

- **The boundary between basins** is fairly sharp. That is, the dynamics is not continuous in the initial conditions, which is to be expected given our step size $h=1$. For dynamical systems in the simple case such boundaries are legitimate.

  In terms of carrying our mathematical machinery over to models, this gives the conclusion that characterising a point by its nearest attractor is incorrect near the boundary.

- **The number of steps to convergence** varies, but it too has structure. And convergence is faster where we have little data. This can be used for OOD detection, and the paper describes how.

- **The decoded attractors** are the simplest digits, zero and one.

## **Part 4. What attractors remember: memorisation versus generalisation**

The central claim of the paper: **attractors are the dictionary the network has learned**, and from it one can see which regime the model is in. If the attractors coincide with training examples, the model
**memorises**. If the attractors have become abstract "building blocks", the model **generalises**.

### **4.1 Proposition 3.2 from the paper**

The formal link goes as follows. Let $\mathcal{Z}^\star$ be a finite dictionary of attractors, $\Pi(z)$ the
projection onto the nearest attractor, and let the decoder $D$ be Lipschitz with constant $L_D$. Then for any test
point $x$ the paper claims:

$$|x - F(x)\|_2^2 \;\le\; \underbrace{\|x - D(\Pi(z))\|_2^2}_{\text{prototype error}} + \underbrace{L_D^2 \|z - \Pi(z)\|_2^2}_{\text{coverage error}}$$

The meaning is this: the reconstruction error splits into two terms. The first is
**how good the prototype is**: whether the decoded version of the nearest attractor is close to the object itself. The second is **how densely the attractors cover the latent space**. Memorisation drives the first term to zero on training points but leaves the coverage insufficient. Generalisation requires the attractors
to be good prototypes and to cover the space densely at the same time.

<details>
<summary><b>Aside: there is a typo in the derivation of this proposition</b></summary>

The derivation itself (Appendix A.3) contains a typo. The step in the proof: we decompose

$$x - D(E(x)) = \underbrace{\big(x - D(\Pi(z))\big)}_{a} + \underbrace{\big(D(\Pi(z)) - D(E(x))\big)}_{b}$$

and then it says "taking norms and applying the Cauchy–Schwarz inequality, we obtain
$\|a + b\|^2 \le \|a\|^2 + \|b\|^2$".

But expanding gives

$$\|a+b\|^2 = \|a\|^2 + 2\langle a, b\rangle + \|b\|^2$$

and the inequality $\|a+b\|^2 \le \|a\|^2 + \|b\|^2$ holds **only if** $\langle a, b\rangle \le 0$.
The correct variants are either without the squares, by the triangle inequality:

$$\|x - F(x)\| \;\le\; \|x - D(\Pi(z))\| + L_D \|z - \Pi(z)\|$$

or with the squares and a factor of 2 (by the inequality $\|a+b\|^2 \le 2\|a\|^2 + 2\|b\|^2$):

$$\|x - F(x)\|^2 \;\le\; 2\|x - D(\Pi(z))\|^2 + 2 L_D^2 \|z - \Pi(z)\|^2$$

The substantive "prototype + coverage" decomposition does not change because of this; the constant does. A check — take $a = b$, then

$$\|a + b\|^2 = \|2a\|^2 = 4\|a\|^2 \;>\; 2\|a\|^2 = \|a\|^2 + \|b\|^2$$

One should use the form with the factor of 2, or the version without the squares — those hold unconditionally.

> Note: I also checked this inequality on our trained model — there are no violations, but here too it matters that our model is a simple one.
</details>

### **4.2 The experimental setup**

Let's look at how training moves as a function of the parameters. The authors use 30 convolutional autoencoders on CIFAR-10, MNIST and FashionMNIST, with bottleneck size
$k$ from 2 to 512, Adam with lr $5 \cdot 10^{-4}$ and weight decay $10^{-4}$, for 500 epochs.

We use: 6 autoencoders on MNIST, $k \in \{2, 4, 8, 16, 32, 128\}$, the same optimiser and
regularisation, but 20 epochs and 10,000 training images — so that everything runs on CPU in reasonable time.

We measure:

- **The memorisation coefficient** $\mathrm{mem}(z^*) = \cos(D(z^*), x_{\text{nearest}})$ — how much
  the decoded attractor resembles the training image closest to it.
- **The test error** — the standard measure of generalisation.
- **The rank of the attractor dictionary**: the number of principal components of the matrix $X^* = D(Z^*)$ explaining 90% of the variance (Appendix B.5). The more directions the attractors cover, the "richer" the dictionary.
- **The number of distinct attractors** — under a rigidly fixed protocol, as we agreed in part 3.

> **Note: typo number two.** In the text of the paper the memorisation coefficient is
> defined by the formula $\mathrm{mem}(z^*) = \min_{x \in \mathcal{X}_{\text{train}}} \cos(D(z^*), x)$,
> while in words it is described as "the cosine similarity of the decoded attractor to its **closest** point in
> the training set". These are different objects: the closest one corresponds to the **maximum** of the cosine, not the minimum.
> The minimum would give the similarity to the most **dissimilar** image and could not equal the reported
> $0.95$–$1.0$. We compute the maximum — that is, we follow the description.

In [ ]:
# ── Null levels for the memorisation coefficient ─────────────────────
# Before celebrating a high cos, we need to know what it gives WITHOUT any memorisation at all.
Ftr_n = F.normalize(Xtr.flatten(1), dim=1)
Fte_n = F.normalize(Xte.flatten(1), dim=1)

S = Ftr_n @ Ftr_n.T
S.fill_diagonal_(-1)
null_train = S.max(1).values.mean().item()
null_test = (Fte_n @ Ftr_n.T).max(1).values.mean().item()
g = torch.Generator().manual_seed(7)
null_noise = (F.normalize(torch.rand(500, 784, generator=g), dim=1) @ Ftr_n.T).max(1).values.mean().item()
del S

print("What the max-cosine to the training set gives on its own:")
print(f"  training image → nearest OTHER training image : {null_train:.3f}")
print(f"  test image → nearest training image: {null_test:.3f}  (this is the null)")
print(f"  uniform noise → nearest training image        : {null_noise:.3f}")

Why do we need the nulls? **Any** MNIST image has a cosine of $\approx 0.86$ to the nearest training one, because MNIST digits are simple (white strokes on a black background), all values are non-negative, and the cosine between any two images is structurally high. Even uniform noise gives $\approx 0.52$.

That is why we will analyse the **difference** from this level and the
**comparison between models**.

In [ ]:
# ── Sweep over the bottleneck size ───────────────────────────────────
def mem_coeff(Xstar):
    """max-cosine of a decoded attractor to the nearest training image."""
    return (F.normalize(Xstar.flatten(1), dim=1) @ Ftr_n.T).max(1).values


def rank90(Xstar):
    """How many principal components of the attractor dictionary explain 90% of the variance."""
    M = Xstar.flatten(1).numpy()
    s = np.linalg.svd(M - M.mean(0), compute_uv=False)
    return int(np.searchsorted(np.cumsum(s ** 2) / np.sum(s ** 2), 0.90) + 1)


KS = [2, 4, 8, 16, 32, 128]
rows, models = [], {}
t_all = time.time()
for k in KS:
    t0 = time.time()
    seed_all(0)
    m = train(AE(k), Xtr, epochs=20, seed=0)
    with torch.no_grad():
        Z0 = m.enc(Xtr[:300])
    zs_k, steps_k = iterate(m, Z0.clone(), n_iter=200, tol=1e-5)
    with torch.no_grad():
        Xstar = m.dec(zs_k)
    mc = mem_coeff(Xstar)
    rows.append(dict(k=k, test_mse=test_mse(m, Xte), mem=mc.mean().item(), mem_sd=mc.std().item(),
                     n_cos=len(dedup(zs_k, 0.99)), n_euc=len(attractor_set(zs_k, 0.01)),
                     rank90=rank90(Xstar), steps_med=float(steps_k.float().median())))
    models[k] = m
    print(f"k={k:>3}  {time.time()-t0:>5.0f} s  " +
          "  ".join(f"{a}={rows[-1][a]:.3f}" if isinstance(rows[-1][a], float) else f"{a}={rows[-1][a]}"
                    for a in ["test_mse", "mem", "n_cos", "rank90"]), flush=True)

df = pd.DataFrame(rows)
print(f"\ntotal {time.time()-t_all:.0f} s\n")
print(df.round(4).to_string(index=False))

In [ ]:
# ── Figure: memorisation versus generalisation (a replication of Figure 2 of the paper) ──
fig, axes = plt.subplots(1, 4, figsize=(17, 3.9))

ax = axes[0]
ax.errorbar(df.k, df.mem, yerr=df.mem_sd, marker="o", capsize=3, color="crimson")
ax.axhline(null_test, ls="--", c="k", lw=1.2, label=f"null: test image ({null_test:.2f})")
ax.axhline(null_noise, ls=":", c="gray", lw=1.2, label=f"null: noise ({null_noise:.2f})")
ax.set_xscale("log", base=2); ax.set_xticks(KS); ax.set_xticklabels(KS)
ax.set_xlabel("$k$ (bottleneck)"); ax.set_ylabel("$\\mathrm{mem}(z^*)$")
ax.set_title("Memorisation coefficient", fontweight="bold", fontsize=11)
ax.legend(fontsize=7); ax.grid(alpha=0.25)

ax = axes[1]
ax.plot(df.k, df.test_mse, marker="s", color="royalblue")
ax.set_xscale("log", base=2); ax.set_xticks(KS); ax.set_xticklabels(KS)
ax.set_xlabel("$k$"); ax.set_ylabel("test MSE")
ax.set_title("Generalisation error", fontweight="bold", fontsize=11); ax.grid(alpha=0.25)

ax = axes[2]
ax.plot(df.k, df.rank90, marker="D", color="seagreen")
ax.set_xscale("log", base=2); ax.set_xticks(KS); ax.set_xticklabels(KS)
ax.set_xlabel("$k$"); ax.set_ylabel("rank at 90% variance")
ax.set_title("Richness of the attractor dictionary", fontweight="bold", fontsize=11); ax.grid(alpha=0.25)

ax = axes[3]
ax.plot(df.k, df.n_cos, marker="o", label="merging by cosine > 0.99")
ax.plot(df.k, df.n_euc, marker="x", ls="--", label="merging by Euclid (1% of median)")
ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xticks(KS); ax.set_xticklabels(KS)
ax.set_xlabel("$k$"); ax.set_ylabel("distinct attractors out of 500")
ax.set_title("Number of attractors\n(and how it depends on the criterion)", fontweight="bold", fontsize=11)
ax.legend(fontsize=7); ax.grid(alpha=0.25)

fig.tight_layout()
plt.show()

In [ ]:
# ── What the attractors themselves look like for different k ─────────
fig, axes = plt.subplots(len(KS), 8, figsize=(9, 1.15 * len(KS)))
for r, k in enumerate(KS):
    m = models[k]
    with torch.no_grad():
        Z0 = m.enc(Xtr[:100])
    zs_k, _ = iterate(m, Z0.clone(), n_iter=200, tol=1e-5)
    keep = dedup(zs_k, 0.99)[:8]
    with torch.no_grad():
        Xs = m.dec(zs_k[keep])
    for c in range(8):
        ax = axes[r, c]; ax.axis("off")
        if c < len(keep):
            ax.imshow(Xs[c, 0], cmap="gray")
    axes[r, 0].set_ylabel(f"k={k}")
    axes[r, 0].axis("on"); axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
fig.suptitle("Decoded attractors $D(z^*)$: from prototypes to textures", fontweight="bold")
fig.tight_layout()
plt.show()

**Results**

**The central claim of the paper reproduced.** The memorisation coefficient falls as $k$ grows
($0.94 \to 0.95 \to 0.53 \to 0.53 \to 0.26 \to 0.20$), while the test error falls monotonically
($0.054 \to 0.0055$). The trade-off "stronger regularisation means more memorisation and worse generalisation" shows up
exactly as in Figure 2 of the paper.

**Taking the nulls into account,** only $k=2$ and $k=4$ ended up above the level of an arbitrary test image ($0.856$).
Already at $k=8$ the coefficient drops to $0.53$ — the level of
**uniform noise** ($0.519$) — and at $k=32$ and $k=128$ it falls **below** the noise level.

So the correct phrasing is not "the attractors became generalised features" but "the decoded
attractors stopped looking like MNIST at all". The figure with the attractors confirms this: from
recognisable prototypes at $k=2$ to uniform blobs at large $k$. In practice this may also be a signal of generalisation.

**The number of attractors jumps around.** By cosine: $3, 6, 8, 34, 5, 1$.
By Euclid: $37, 56, 11, 94, 5, 300$. Note $k=128$: **1 versus 300**. All three hundred
stopping points are distinct in Euclidean distance and yet pairwise have $\cos > 0.99$ — that is,
they lie almost along a single ray and differ mainly in length. This is a disagreement between the metrics.

**The rank of the attractor dictionary** ($2, 2, 3, 5, 2, 2$) does not grow monotonically
(as shown in Appendix B.5).

**Hypotheses.** In our simple experiment we train for 20 epochs on 10,000 images; in the paper it is 500 epochs on full datasets.
Perhaps our field is simply too simple, and the map $f$ has stayed close to its
strongly contractive initial state. Let's check this directly — for $k=128$, at 50 epochs.

In [ ]:
# ── How much the field depends on the training budget ────────────────
seed_all(0)
ae128_long = train(AE(128), Xtr, epochs=50, seed=0)

comp = []
for name, m in [("k=128, 20 epochs", models[128]), ("k=128, 50 epochs", ae128_long)]:
    with torch.no_grad():
        Z0 = m.enc(Xtr[:300])
    zs_c, st_c = iterate(m, Z0.clone(), n_iter=200, tol=1e-5)
    with torch.no_grad():
        Xs_c = m.dec(zs_c)
    comp.append(dict(model=name, test_mse=round(test_mse(m, Xte), 4),
                     mem=round(mem_coeff(Xs_c).mean().item(), 3),
                     by_cosine=len(dedup(zs_c, 0.99)), by_euclid=len(attractor_set(zs_c, 0.01)),
                     rank90=rank90(Xs_c), steps_median=int(st_c.float().median())))
print(pd.DataFrame(comp).to_string(index=False))

The hypothesis holds. Over 30 extra epochs:

- the number of cosine-distinct attractors grew from **1 to 28**;
- the rank of the dictionary from **2 to 7**;
- the median number of steps to convergence from **26 to 63**, that is, the map became noticeably
  **less** contractive.

So the degeneracy of our field is a consequence of the training budget.
The Euclidean count, meanwhile, stayed at 300 as it was: it is saturated from the very start and measures the scatter of the stopping
points rather than the structure of the field.

> **The practical takeaway.** Before saying anything about the
> attractors of a model, one has to make sure the field is non-trivial. The simplest check is
> the **median number of steps to convergence**: if it is 26 steps out of 200, you are not looking at rich
> dynamics but at an almost constant map, and any conclusions about a "dictionary of features" will be
> conclusions about a single point.

## **Part 5. How the field is born over the course of training**

So far we have been comparing **different** models. The paper shows (Figure 3) that the same
"memorisation → generalisation" transition happens inside **one** model over the course of training. The claims are:

1. At initialisation there is exactly one attractor — everything collapses to the origin (we measured this
   in part 2: $\sigma_{\text{out}}/\sigma_{\text{in}} \approx 0.04$).
2. In the first epochs the model **memorises**: the memorisation coefficient is high, the test error is large.
3. Then the memorisation coefficient drops, the test error drops too, and the number of distinct attractors
   grows and reaches a plateau.
4. Attractors computed **from noise** gradually converge to the attractors computed from
   the training data — that is, the dictionary becomes the same no matter where you start from.
5. But the **trajectories** to those attractors stay distinguishable. That is exactly why OOD detection
   works later on: the attractor is the same, the path to it is different.

Let's replicate all five claims on a model with $k = 64$, logging statistics every 5 epochs.

In [ ]:
# ── Tools for trajectories (they will also be handy in part 6) ───────
@torch.no_grad()
def traj_score(model, Z0, A_star, n_iter=30):
    """
    score(z) = (1/N) * sum_t d(z_t, Z*_train) — the distance to the set of training
    attractors averaged along the trajectory (the formula from section 4.2 of the paper).
    """
    model.eval()
    z, acc = Z0.clone(), torch.zeros(len(Z0))
    for _ in range(n_iter):
        acc += torch.cdist(z, A_star).min(1).values
        z = model.f(z)
    return (acc / n_iter).numpy()


def chamfer_cos(A, B):
    """Symmetric cosine similarity between two sets of points (Chamfer)."""
    S = F.normalize(A, dim=1) @ F.normalize(B, dim=1).T
    return 0.5 * (S.max(1).values.mean() + S.max(0).values.mean()).item()


def fpr_at_95tpr(s_id, s_ood):
    """The share of OOD objects accepted as in-distribution at a threshold that lets 95% of ID through."""
    return float((s_ood <= np.quantile(s_id, 0.95)).mean())

In [ ]:
# ── Training dynamics (a replication of Figure 3 of the paper) ───────
seed_all(0)
ae64 = AE(64)
noise_img = torch.rand(150, 1, 28, 28, generator=torch.Generator().manual_seed(11))
hist = []

def log_step(model, ep):
    if ep % 5 and ep != 0:
        return
    model.eval()
    with torch.no_grad():
        Ztr0, Zte0, Zn0 = model.enc(Xtr[:150]), model.enc(Xte[:150]), model.enc(noise_img)
    A_tr, _ = iterate(model, Ztr0.clone(), 80, 1e-5)
    A_te, _ = iterate(model, Zte0.clone(), 80, 1e-5)
    A_n, _ = iterate(model, Zn0.clone(), 80, 1e-5)
    with torch.no_grad():
        Xs = model.dec(A_tr)
    A_uniq = A_tr[dedup(A_tr, 0.99)]
    hist.append(dict(
        ep=ep, test=test_mse(model, Xte), mem=mem_coeff(Xs).mean().item(),
        n_tr=len(dedup(A_tr, 0.99)) / 150, n_te=len(dedup(A_te, 0.99)) / 150,
        n_noise=len(dedup(A_n, 0.99)) / 150, cham=chamfer_cos(A_tr, A_n),
        fpr95=fpr_at_95tpr(traj_score(model, Ztr0, A_uniq), traj_score(model, Zn0, A_uniq)),
    ))
    print({a: (round(v, 4) if isinstance(v, float) else v) for a, v in hist[-1].items()}, flush=True)
    model.train()

t0 = time.time()
train(ae64, Xtr, epochs=31, seed=0, log=log_step)
dyn = pd.DataFrame(hist)
print(f"\n{time.time()-t0:.0f} s")

In [ ]:
# ── Dynamics plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.plot(dyn.ep, dyn.mem, "o-", color="crimson", label="memorisation coefficient")
ax.axhline(null_test, ls="--", c="k", lw=1, label=f"null level ({null_test:.2f})")
ax.set_xlabel("epoch"); ax.set_ylabel("$\\mathrm{mem}(z^*)$", color="crimson")
ax2 = ax.twinx()
ax2.plot(dyn.ep, dyn.test, "s--", color="royalblue", label="test MSE")
ax2.set_yscale("log"); ax2.set_ylabel("test MSE", color="royalblue")
ax.set_title("First memorise, then generalise", fontweight="bold", fontsize=11)
ax.legend(fontsize=7, loc="upper right"); ax.grid(alpha=0.25)

ax = axes[1]
for col, lb, c in [("n_tr", "from training data", "royalblue"), ("n_te", "from test data", "crimson"),
                   ("n_noise", "from noise", "goldenrod")]:
    ax.plot(dyn.ep, dyn[col], "o-", color=c, label=lb)
ax.set_xlabel("epoch"); ax.set_ylabel("share of distinct attractors")
ax.set_title("The number of attractors grows with training", fontweight="bold", fontsize=11)
ax.legend(fontsize=8); ax.grid(alpha=0.25)

ax = axes[2]
ax.plot(dyn.ep, dyn.cham, "o-", color="seagreen", label="dictionary similarity (noise vs data)")
ax.set_xlabel("epoch"); ax.set_ylabel("Chamfer cos", color="seagreen")
ax3 = ax.twinx()
ax3.plot(dyn.ep, dyn.fpr95, "s--", color="purple", label="FPR95 of trajectories")
ax3.set_ylabel("FPR95 (lower is better)", color="purple")
ax.set_title("The dictionaries converge, the trajectories do not", fontweight="bold", fontsize=11)
ax.legend(fontsize=7, loc="lower right"); ax.grid(alpha=0.25)

fig.tight_layout()
plt.show()

### **5.1 What reproduced and what did not**

**Claim 1 reproduced perfectly.** At epoch zero the share of distinct attractors equals $1/150$ —
that is, all 150 starting points, wherever they come from (training data, test data, noise), arrive
at **one and the same** point. An untrained network is a single global attractor.

**Claim 2 reproduced halfway.** The memorisation coefficient at epoch zero is indeed
high ($0.92$) and falls to $0.23$ by the fifth epoch — but measured against the null level of $0.856$,
the excess is only $+0.06$.

**Claims 3 and 4 did not reproduce.** The share of distinct attractors
jumps around for us ($0.007 \to 0.087 \to 0.047 \to 0.473 \to 0.187 \to 0.020$).
And the Chamfer similarity between the dictionaries from noise and from data equals $1.000$ already at epoch zero and never drops
below $0.81$ over the whole run. Both metrics are **cosine-based**, and in $\mathbb{R}^{64}$ all the attractors of our model
point in almost the same direction — the same anisotropy that in part 4 made cosine merging
collapse 300 distinct points into a single "attractor".

**Claim 5 (trajectories become distinguishable, FPR95 falls) reproduced with the opposite sign:**
our FPR95 **grows** from $0.00$ to $1.00$, whereas it should fall. Probably again because of the structure of the data — we took uniform noise as OOD. A uniformly noisy $28\times 28$ image, after the encoder, lands **in the centre** of the latent cloud: it has
neither pronounced strokes nor structure, and its code is close to the mean code. Which means its trajectory
passes **closer** to the attractors than the trajectory of a real digit, and the score flips by
construction. In the paper the OOD sets are SUN397, Places365, Texture, iNaturalist, that is,
**natural images**, while Gaussian noise is used only as a source of starting points for
the attractor search, never as an OOD class.

> The moral: **"OOD" is a task that demands a careful setup.** An object that is more complex than
> the training distribution and an object that is simpler than it behave in opposite ways in the
> vector field. Any metric of the form "distance to the attractors" implicitly assumes
> the first case.

## **Part 6. What can be done with this field**

Theory and diagnostics are fine, but the paper offers two genuinely applied uses. We will
reproduce both.

### **6.1 Data-free probing: what sits in the weights when there is no data at all**

The setup: we have a pretrained model. We do not have the data it was trained on. The question: can we get anything meaningful out of the weights?

The paper's idea: sample $Z_n \sim \mathcal{N}(0, I)$, iterate until attractors are found, declare the resulting set a dictionary of signals, and then see how sparsely real data can be coded with it. For the authors this is the Stable Diffusion AE, 4096 noise points and six datasets, and the result: the attractors win on every dataset.

I was not able to reproduce this on the simple example considered here, and my hypothesis is, again, the poverty of the model's dictionary and the simplicity of the setup.

### **6.2 OOD detection from trajectories**

The second task is to determine whether an object came from the distribution the model was trained on.

The paper's idea: an OOD object has two ways of giving itself away, and we check both:

1. Its trajectory **does not converge** to the attractors of the training data — that is, it is in a different
   basin altogether.
2. Even if the basin is shared, it **converges differently** — usually faster, because outside the support of
   the data the field is stronger (we saw this in part 2: $\|J_f\| \approx 0.25$ far away and $\approx 1$ on the data).

To catch both cases, one takes **the whole trajectory**, not just its endpoint:

$$\mathrm{score}(z) = \frac{1}{N} \sum_{z_i \in \pi(z)} d\big(z_i, \mathcal{Z}^*_{\text{train}}\big),
\qquad \pi(z) = [z_0, \dots, z_N]$$

The resulting score is compared against **KNN in feature space** — the mean distance to the $K$
nearest latents of the training set (an adaptation of Sun et al., 2022):

$$\mathrm{score}_{\mathrm{KNN}}(z) = \frac{1}{K} \sum_{u \in \mathrm{NN}_K(z,\; \mathcal{Z}_{\text{train}})} d(z, u)$$

The difference between them: the trajectory score
measures the distance to the **attractors** and along the entire path, while KNN measures it to **the training
latents themselves** and at a single point. The attractors sit in the centre of the cloud, the training latents are spread over all of it.

Both quality metrics are computed as classification quality under one labelling: ID gets label $0$, OOD gets label $1$.
**AUROC** is the probability that a random OOD object gets a higher score than a random ID object
($0.5$ — the method distinguishes nothing). **FPR95** — we take the threshold that lets 95% of the in-distribution objects through and see
what share of the alien ones slipped through with them; the smaller, the better.

Let's look at this experiment. We take MNIST test as ID, FashionMNIST and uniform noise as OOD, and our $k=32$ model.

In [ ]:
# ── OOD detection: trajectories versus KNN ───────────────────────────
Xf_tr, _, Xf_te, _ = load(10, 2000, fashion=True)      # FashionMNIST as OOD

ae32 = models[32]
with torch.no_grad():
    Ztr32 = ae32.enc(Xtr[:1000])
A_tr32, _ = iterate(ae32, Ztr32.clone(), 300, 1e-5)
A_uniq32 = A_tr32[dedup(A_tr32, 0.99)]
print(f"training attractors: {len(A_tr32)} points → {len(A_uniq32)} distinct")

with torch.no_grad():
    Z_id = ae32.enc(Xte)
    Z_fashion = ae32.enc(Xf_te)
    Z_unif = ae32.enc(torch.rand(2000, 1, 28, 28, generator=torch.Generator().manual_seed(3)))

sets = {"MNIST test (ID)": Z_id, "FashionMNIST": Z_fashion, "uniform noise": Z_unif}
traj = {n: traj_score(ae32, Z, A_uniq32) for n, Z in sets.items()}

K = 50
knn = {n: torch.cdist(Z, Ztr32).topk(K, largest=False).values.mean(1).numpy() for n, Z in sets.items()}

print(f"\n{'OOD set':>18} | {'trajectories':^19} | {'KNN (K=50)':^19}")
print(f"{'':>18} | {'AUROC':>8} {'FPR95':>9} | {'AUROC':>8} {'FPR95':>9}")
ood_rows = []
for ood in ["FashionMNIST", "uniform noise"]:
    y = np.r_[np.zeros(len(traj["MNIST test (ID)"])), np.ones(len(traj[ood]))]
    a_t = roc_auc_score(y, np.r_[traj["MNIST test (ID)"], traj[ood]])
    a_k = roc_auc_score(y, np.r_[knn["MNIST test (ID)"], knn[ood]])
    f_t = fpr_at_95tpr(traj["MNIST test (ID)"], traj[ood])
    f_k = fpr_at_95tpr(knn["MNIST test (ID)"], knn[ood])
    print(f"{ood:>18} | {a_t:>8.3f} {f_t:>9.3f} | {a_k:>8.3f} {f_k:>9.3f}")
    ood_rows.append(dict(ood=ood, auroc_traj=a_t, fpr_traj=f_t, auroc_knn=a_k, fpr_knn=f_k))

Let's take a look at the 5 attractors we are measuring everything against.

In [ ]:
# ── What are the five points everything is measured against?
# The whole trajectory score is the distance to A_uniq32. Let's look at them:
# how many starts arrived at each, where they lie and what they resemble.
lab32 = torch.cdist(A_tr32, A_uniq32).argmin(1)
sizes32 = torch.bincount(lab32, minlength=len(A_uniq32))

with torch.no_grad():
    X_star32 = ae32.dec(A_uniq32)
sims32 = F.normalize(X_star32.flatten(1), dim=1) @ F.normalize(Xtr.flatten(1), dim=1).T
best32 = sims32.max(1)

print(f"{'attractor':>10} {'starts':>9} {'norm':>8} {'cos to nearest training image':>31}")
for i in range(len(A_uniq32)):
    print(f"{'z*_' + str(i + 1):>10} {sizes32[i].item():>9} "
          f"{A_uniq32[i].norm():>8.1f} {best32.values[i]:>31.3f}")
print(f"\nfor comparison: median norm of the data latents {Ztr32.norm(dim=1).median():.1f}; "
      f"null level of the max-cosine (unseen image → training image) on MNIST ≈ 0.86")

fig, ax = plt.subplots(2, len(A_uniq32), figsize=(2.1 * len(A_uniq32), 4.4))
for i in range(len(A_uniq32)):
    ax[0, i].imshow(X_star32[i, 0], cmap="gray"); ax[0, i].axis("off")
    ax[0, i].set_title(f"$z^*_{i + 1}$\n{sizes32[i].item()} starts", fontsize=9)
    j = int(best32.indices[i])
    ax[1, i].imshow(Xtr[j, 0], cmap="gray"); ax[1, i].axis("off")
    ax[1, i].set_title(f"closest: {ytr[j]}\ncos = {best32.values[i]:.2f}", fontsize=8)
fig.suptitle("The attractors the OOD score is measured against (top) and the training images closest to them (bottom)",
             fontweight="bold", fontsize=11)
fig.tight_layout()
plt.show()

The decoded attractors are blurry blobs and textures,
with a cosine to the nearest training image of $0.15$–$0.40$ against a null level of $0.86$. That is, they are less like MNIST than uniform noise is (whose null is $0.52$).

> From here we see that the score should only give a correct result if the attractors describe the data and lie **inside** the cloud. If that is not true, things come out crooked.

In [ ]:
# ── Figure: distributions of the scores ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, (scores, title) in zip(axes, [(traj, "trajectory score"), (knn, "KNN in latent space")]):
    for n, c in zip(sets, ["seagreen", "crimson", "slategray"]):
        v = scores[n]
        ax.hist(v, bins=60, alpha=0.55, label=n, color=c, density=True)
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_xlabel("score (higher means «more alien»)"); ax.set_ylabel("density")
    ax.legend(fontsize=8)
fig.suptitle("Separability of ID and OOD: trajectories versus KNN", fontweight="bold")
fig.tight_layout()
plt.show()

**What we see.** The horizontal axis is the value of the score, the vertical one is density.
A good picture should look like two non-overlapping humps: green (ours) on the left, coloured (aliens) on the
right. The less they overlap, the higher the AUROC.

- **On the left, the trajectory score.** The green and grey humps sit on top of each other — by this score the noise
  is indistinguishable from real digits. The red one is shifted
  right and has a separate peak at the right edge: part of FashionMNIST travels very far from
  the attractors and stays there.
- **On the right, KNN.** The grey hump has turned into a narrow peak standing to the right of the green one — the noise separates
  beautifully. The red one is wide and overlaps the green one heavily.

> **Why the densities look so different.** The two scores are two different quantities in different units: on the left, the mean distance to five attractors along thirty steps; on the right, the mean distance to the fifty nearest training latents at a single point.
> Their ranges differ (0–95 versus 15–70), and the plotting normalises the area of each hump to one.
> That is why a narrow distribution is drawn tall and a wide one low, and the heights of the bars cannot be compared between the panels — only the shape and the overlap can.
> The narrow grey peak on the right follows from the fact that two thousand uniformly noisy images are statistically identical: they have neither strokes nor structure, so they encode to almost the same point — and their distances to the training set come out almost the same.
> On the left the same noise is smeared wider, because the trajectory score averages over thirty steps, and along that path the points have time to drift apart.
> The scale does not affect the metrics themselves: both AUROC and FPR95 depend only on the ordering of the scores, not on their absolute values.

**The numbers:**

**FashionMNIST (OOD).** Trajectory score: AUROC $0.828$, FPR95 $0.374$.
KNN: AUROC $0.821$, FPR95 $0.528$. On AUROC the methods are essentially level, but on FPR95 the trajectories are noticeably better: $0.374$ against $0.528$.

**Uniform noise.** Trajectory score: AUROC $0.729$, FPR95 $0.807$. KNN: AUROC $0.943$, FPR95 $0.517$. Here the trajectories **lose** by a wide margin.

The reason is most likely simplicity again: a uniformly noisy $28\times 28$ image is an object **simpler** than the training distribution. For the task, such data has neither strokes nor structure, and the encoder sends it to the **centre** of the latent cloud.

And the centre is exactly where the attractors live. So the trajectory of the noise passes **closer** to the training attractors than the trajectory of a real digit, and the score, built as "distance to the attractors", flips its sign. Note also how frightening the attractors themselves turned out to be: they are not informative, even though the nearest points are distinct and correspond to specific classes.

KNN does not have this problem: it measures the distance to **the training latents themselves** rather than to the attractors, and the training latents lie not in the centre but in a shell around it.

> **A note:**
> the trajectory score answers not quite the question "is this object from a different distribution?" but the question
> "is this object **further from the modes** of the learned distribution?". For natural OOD sets these two questions coincide.
> For degenerate inputs — noise, constant images, heavily blurred images (and most likely for adversarial ones too) — they are
> opposite, and the method may give the wrong answer.

## **Part 7. Limits of applicability**

So, we have worked through an experiment that lets one study the model's internal picture of the world using ideas from dynamical systems. The approach is not universal and is valid only for autoencoders; a discriminative model — a classifier, an encoder-only model, a next-token predictor — does not have it.

The paper covers this too, and almost successfully, but with limits on how the experiments may be interpreted. Specifically, it describes that:

- one can train a decoder on top of a frozen encoder (in Appendix B.8.1 this is done for DINOv2 and SigLIP2);
- one can work in the output space, defining the field as the residual;
- one can iterate the entire residual stream (in Appendix B.8.2 — for Qwen3-0.6B and SmolLM2-1.7B).

Among the other limitations, it is also fair to say that while we were running the experiment in a simple setup, we did not get full reproducibility (although the core geometric patterns reproduced beautifully, the practical ones came out not quite accurate or meaningful, owing to the simplicity of the data).

The thing to take away from here is that when you try to repeat strong results — make sure the setup you have and the one being considered are genuinely close.

That's all from me, and well done to you!
Thank you for reading to the end! I hope you will repeat something of this on your own, or simply enjoyed the beauty of the theory.

Until next time, friends!

If you enjoyed it, as always I invite you to:
- [Telegram Blog, ru](https://t.me/jdata_blog)
- [Other tutorials]()
- [LinkedIn](https://www.linkedin.com/in/sabrina-sadiekh)